In [1]:
# Calculate all FWI components using the pre-formatted climatex data 

import numpy as np
import pandas as pd
import math
# from glob import glob
import cfgrib
import xarray as xr
import pygrib
import datetime
from datetime import timedelta
import time
import timezonefinder, pytz

from zoneinfo import ZoneInfo
from tqdm import tqdm

In [3]:
# Load pre-formatted climatex data (one model at a time)
folder = r"/Users/cpearson/Projects/fwf/data/timezone/"
climatexdata = xr.open_zarr(folder, 'tzone_wrf4_d03-ST.zarr', consolidated=False) 
climatexdata

<xarray.Dataset>
Dimensions:      (south_north: 702, west_east: 558)
Coordinates:
    XLAT         (south_north, west_east) float32 dask.array<chunksize=(351, 279), meta=np.ndarray>
    XLONG        (south_north, west_east) float32 dask.array<chunksize=(351, 279), meta=np.ndarray>
  * south_north  (south_north) float64 -4.124e+06 -4.121e+06 ... -2.021e+06
  * west_east    (west_east) float64 -3.082e+06 -3.079e+06 ... -1.411e+06
Data variables:
    Zone         (south_north, west_east) int64 dask.array<chunksize=(176, 279), meta=np.ndarray>
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  Time Zone Offsets From UTC
    pyproj_srs:   +proj=stere +lat_0=90 +lat_ts=60 +lon_0=-90 +x_0=0 +y_0=0 +...
    stagger:      
    units:        hours_

In [4]:
### Functions for Drying Factor and Day Length
    # Ref: https://github.com/buckinha/pyfwi/blob/master/pyFWI/FWIFunctions.py 
    # Drying Factor, Day Length

def DryingFactor(Latitude, Month):
#def DryingFactor(ds):
#    Latitude = ds.latitude
#    Month = ds.time.dt.month.values

    LfN = [-1.6, -1.6, -1.6, 0.9, 3.8, 5.8, 6.4, 5.0, 2.4, 0.4, -1.6, -1.6]
    LfS = [6.4, 5.0, 2.4, 0.4, -1.6, -1.6, -1.6, -1.6, -1.6, 0.9, 3.8, 5.8]

  #  if Latitude > 0:
  #      retVal = LfN[Month]
  #  elif Latitude <= 0.0:
  #      retVal = LfS[Month]
  #  return retVal

    #to make compatible with xarray:
    retVal = xr.where(Latitude>0, LfN[Month-1], LfS[Month-1])
    return retVal

def DayLength(Latitude, MONTH):
    #Approximates the length of the day given month and latitude

    DayLength46N = [ 6.5,  7.5,  9.0, 12.8, 13.9, 13.9, 12.4, 10.9,  9.4,  8.0,  7.0,  6.0]
    DayLength20N = [ 7.9,  8.4,  8.9,  9.5,  9.9, 10.2, 10.1,  9.7,  9.1,  8.6,  8.1,  7.8]
    DayLength20S = [10.1,  9.6,  9.1,  8.5,  8.1,  7.8,  7.9,  8.3,  8.9,  9.4,  9.9, 10.2]
    DayLength40S = [11.5, 10.5,  9.2,  7.9,  6.8,  6.2,  6.5,  7.4,  8.7, 10.0, 11.2, 11.8]

    #retVal = None
    #if Latitude<= 90 and Latitude > 33:
    #    retVal = DayLength46N[MONTH-1]
    #elif Latitude <= 33 and Latitude > 0.0:
    #    retVal = DayLength20N[MONTH-1]
    
    #elif Latitude <= 0.0 and Latitude > -30.0:
    #    retVal = DayLength20S[MONTH-1]
    #elif Latitude <= -30.0 and Latitude >= -90.0:
    #    retVal = DayLength40S[MONTH-1]

    #if retVal==None:
    #    raise InvalidLatitude(Latitude)

    # Make compatible with xarray and ignore latitudes in the southern hemisphere:
    retVal = xr.where(Latitude>33, DayLength46N[MONTH-1], DayLength20N[MONTH-1])
    
    return retVal

In [8]:
climatexdata['XLONG'].values

array([[-126.77383 , -126.74706 , -126.72029 , ..., -108.96395 ,
        -108.92666 , -108.88935 ],
       [-126.79381 , -126.76706 , -126.74027 , ..., -108.97677 ,
        -108.93945 , -108.90213 ],
       [-126.813835, -126.787056, -126.760284, ..., -108.9896  ,
        -108.95227 , -108.914925],
       ...,
       [-146.67094 , -146.64532 , -146.61967 , ..., -124.95969 ,
        -124.90268 , -124.84559 ],
       [-146.7099  , -146.68428 , -146.65865 , ..., -124.99954 ,
        -124.94251 , -124.8854  ],
       [-146.74889 , -146.72328 , -146.69766 , ..., -125.03949 ,
        -124.98243 , -124.925285]], dtype=float32)

In [5]:
# Calculate the Daylength 

datasets = []
longitudes = climatexdata['XLON'].values
timelength = len(climatexdata['time'].values)

for i in range(timelength): #(30):
        for j, lon in enumerate(longitudes):
            ds = climatexdata.isel(time=i)
            month = ds.time.dt.month.values
            dldata = DayLength(ds.lat, month)
        
            dldata = dldata.expand_dims(time=[i], longitude=[lon])
            dldata = dldata.rename("daylength") # Need to do this in order to save in xarray
            datasets.append(dldata)

# Save the daylength factor as an xarray with dimensions: time, latitude, and longitude
daylength_all = xr.combine_by_coords(datasets)
daylength_all

KeyError: 'lon'

In [44]:
daylength_all = daylength_all.rename({'longitude': 'lon'})

In [45]:
daylength_all.isel(time=3000).daylength.values

array([[9., 9., 9., ..., 9., 9., 9.],
       [9., 9., 9., ..., 9., 9., 9.],
       [9., 9., 9., ..., 9., 9., 9.],
       ...,
       [9., 9., 9., ..., 9., 9., 9.],
       [9., 9., 9., ..., 9., 9., 9.],
       [9., 9., 9., ..., 9., 9., 9.]])

In [46]:
# Save the df and daylength for these time steps - for future use in the calculations
    
# Save to zarr
daylength_all.to_zarr(r"C:\Users\minal\Documents\UBC\GRID2\daylength_climatex_2004-2014.zarr", mode="w")

In [49]:
# Calculate the Drying Factor
datasets = []
longitudes = climatexdata['lon'].values
timelength = len(climatexdata['time'].values)

for i in range(timelength): #(30):
        for j, lon in enumerate(longitudes):
            ds = climatexdata.isel(time=i)
            month = ds.time.dt.month.values
            dldata = DryingFactor(ds.lat, month)
        
            dldata = dldata.expand_dims(time=[i], longitude=[lon])
            dldata = dldata.rename("dryingfactor") # Need to do this in order to save in xarray
            datasets.append(dldata)

# Save the drying factor as an xarray with dimensions: time, latitude, and longitude
df_all = xr.combine_by_coords(datasets)
df_all 

<xarray.Dataset> Size: 180MB
Dimensions:       (time: 4018, longitude: 112, lat: 50)
Coordinates:
  * time          (time) int32 16kB 0 1 2 3 4 5 ... 4013 4014 4015 4016 4017
  * longitude     (longitude) float64 896B -140.1 -139.9 ... -112.6 -112.4
  * lat           (lat) float64 400B 47.88 48.12 48.38 ... 59.62 59.88 60.12
Data variables:
    dryingfactor  (time, longitude, lat) float64 180MB -1.6 -1.6 ... -1.6 -1.6

In [50]:
# Match coord names 
df_all = df_all.rename({'longitude': 'lon'})

In [51]:
df_all.isel(time=3000).dryingfactor.values

array([[-1.6, -1.6, -1.6, ..., -1.6, -1.6, -1.6],
       [-1.6, -1.6, -1.6, ..., -1.6, -1.6, -1.6],
       [-1.6, -1.6, -1.6, ..., -1.6, -1.6, -1.6],
       ...,
       [-1.6, -1.6, -1.6, ..., -1.6, -1.6, -1.6],
       [-1.6, -1.6, -1.6, ..., -1.6, -1.6, -1.6],
       [-1.6, -1.6, -1.6, ..., -1.6, -1.6, -1.6]])

In [52]:
 # Save to zarr
df_all.to_zarr(r"C:\Users\minal\Documents\UBC\GRID2\df_climatex_2004-2014.zarr", mode="w")

In [81]:
## Load df and daylength if they have been previously calculated
daylength_all = xr.open_zarr(folder, 'daylength_climatex_2004-2014.zarr') 
df_all = xr.open_zarr(folder, 'df_climatex_2004-2014.zarr') 

C:\Users\minal\AppData\Local\Temp\ipykernel_20748\1308543423.py:2: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  daylength_all = xr.open_zarr(folder, 'daylength_cmip_2004-2014.zarr')
C:\Users\minal\AppData\Local\Temp\ipykernel_20748\1308543423.py:3: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr

In [83]:
df_all.isel(time=500).dryingfactor.values

array([[3.8, 3.8, 3.8, ..., 3.8, 3.8, 3.8],
       [3.8, 3.8, 3.8, ..., 3.8, 3.8, 3.8],
       [3.8, 3.8, 3.8, ..., 3.8, 3.8, 3.8],
       ...,
       [3.8, 3.8, 3.8, ..., 3.8, 3.8, 3.8],
       [3.8, 3.8, 3.8, ..., 3.8, 3.8, 3.8],
       [3.8, 3.8, 3.8, ..., 3.8, 3.8, 3.8]])

In [84]:
# Chris' FWI functions 
    # from https://github.com/cerodell/fwf/blob/ams-fwi/utils/fwi.py 

"""########################################################################"""
"""###################### Fine Fuels Moisture Code ########################"""
"""########################################################################"""

def solve_ffmc(ds, F):
    
    W, T, H, r_o = (ds.sfcWind, ds.tasmax, ds.hurs, ds.pr)
    
    #~
    # Make sure no RH > 100 or RH < 0
    H = xr.where(ds.hurs > 100, 100, xr.where(ds.hurs < 0, 0, ds.hurs))

    # #Eq. 1
    m_o = 147.2 * (101 - F) / (59.5 + F)
    ########################################################################
    ### Solve for the effective raut
    r_f = xr.where(r_o > 0.5, (r_o - 0.5), xr.where(r_o < 1e-7, 1e-5, r_o))
    ########################################################################
    ### (1) Solve the Rautfagner 1985 (m_r)
    m_o = xr.where(
        m_o <= 150,
        m_o + (42.5 * r_f * np.exp((-100 / (251 - m_o))) * (1 - np.exp((-6.93 / r_f)))),
        m_o
        + (42.5 * r_f * np.exp((-100 / (251 - m_o))) * (1 - np.exp((-6.93 / r_f))))
        + (0.0015 * np.power((m_o - 150), 2) * np.power(r_f, 0.5)),
    )

    #m_o = np.where(m_o > 250, 250, np.where(m_o < 0, 0.0, m_o))
    m_o = xr.where(m_o > 250, 250, m_o)
    m_o = xr.where(m_o < 0, 0.0, m_o)
        # replacing np.where with xr.where 

    ########################################################################
    ### (2a) Solve Equilibrium Moisture content for dry

    E_d = (
        0.942 * np.power(H, 0.679)
        + 11 * np.exp((H - 100) / 10)
        + 0.18 * (21.1 - T) * (1 - np.exp((-0.115 * H)))
    )

    ########################################################################
    ### (2b) Solve Equilibrium Moisture content for wett

    E_w = (
        0.618 * (np.power(H, 0.753))
        + 10 * np.exp((H - 100) / 10)
        + 0.18 * (21.1 - T) * (1 - np.exp((-0.115 * H)))
    )
    ########################################################################
    ### (3a) ate step to k_d (k_a)
    k_a = 0.424 * (1 - np.power(H / 100, 1.7)) + 0.0694 * (np.power(W, 0.5)) * (
        1 - np.power(H / 100, 8)
    )

    ########################################################################
    ### (3b) Log dryfor hourly computation, log to base 10 (k_d)
    k_d = k_a * 0.581 * np.exp(0.0365 * T)

    ########################################################################
    ### (4a) ate steps to k_w (k_b)
    k_b = 0.424 * (1 - np.power(((100 - H) / 100), 1.7)) + 0.0694 * np.power(W, 0.5) * (
        1 - np.power(((100 - H) / 100), 8)
    )

    ########################################################################
    ### (4b)  Log wet for hourly computation, log to base 10 (k_w)
    k_w = k_b * 0.581 * np.exp(0.0365 * T)

    ########################################################################
    ### (5a) ate dry moisture code (m_d)
    m_d = E_d + (m_o - E_d) * 10 ** (-k_d)

    ########################################################################
    ### (5b) ate wet moisture code (m_w)
    m_w = E_w - (E_w - m_o) * 10 ** (-k_w)

    ########################################################################
    ### (5c) combwet, neutral moisture codes
    m = xr.where(m_o > E_d, m_d, m_w)
    m = xr.where((E_d >= m_o) & (m_o >= E_w), m_o, m)
    # m_o = xr.DataArray(m, name="m_o", dims=("temp", "wind", "rh", "precip"))

    # ds["m_o"] = m_o

    ########################################################################
    ### (6) Solve for FFMC
    F = (59.5 * (250 - m)) / (147.2 + m)  ## Van 1985
    # F = xr.DataArray(F, name="F", dims=("temp", "wind", "rh", "precip"))
    
    #ds["F"] = F
    #return ds
    return F


"""########################################################################"""
""" ########################## Duff Moisture Code #########################"""
"""########################################################################"""


def solve_dmc(ds, P, L_e):

    W, T, H, r_o, P_o, L_e = (ds.sfcWind, ds.tasmax, ds.hurs, ds.pr, P, L_e.daylength)

    #~
    # Make sure no RH > 100 or RH < 0
    #H = xr.where(ds.hurs > 100, 100, xr.where(ds.hurs < 0, 0, ds.hurs))
    H = xr.where(H > 100, 100, xr.where(H < 0, 0, H))


    #zero_full = np.zeros_like(ds.T)
    zero_full = xr.zeros_like(ds.tasmax)
    
    ##  Constrain temp
    ##    - The log drying rate K is proportional to temperature, becoming negligible at about -1.1°C (Van Wagner 1985) .
    #T = np.where(T < -1.1, -1.1, T)
    T = xr.where(T < -1.1, -1.1, T)

    ########################################################################
    ### (11) Solve for the effective rain (r_e)
    r_e = (0.92 * r_o) - 1.27

    ########################################################################
    ### (12) NOTE Alteratered for more accurate calculation (Lawson 2008)
    M_o = 20 + 280 / np.exp(0.023 * P_o)

    ########################################################################
    ### (13a) Solve for coefficients b where P_o <= 33 (b_low)
    b_low = xr.where(P_o <= 33, 100 / (0.5 + 0.3 * P_o), zero_full)

    ########################################################################
    ### (13b) Solve for coefficients b where 33 < P_o <= 65 (b_mid)

    b_mid = xr.where((P_o > 33) & (P_o <= 65), 14 - 1.3 * np.log(P_o), zero_full)

    ########################################################################
    ### (13c) Solve for coefficients b where  P_o > 65 (b_high)

    b_high = xr.where(P_o > 65, 6.2 * np.log(P_o) - 17.2, zero_full)
    ########################################################################
    ### Combine (13a 13b 13c) for coefficients b
    b = b_low + b_mid + b_high

    ########################################################################
    ### (14) Solve for moisture content
    M_r = M_o + 1000 * r_e / (48.77 + b * r_e)

    ########################################################################
    ### (15) Duff moisture code (P_r) Alteration more accurate calculation (Lawson 2008)
    P_r = 43.43 * (5.6348 - np.log(M_r - 20))

    ### Mina's additions for errors
    # Safe log
    #M_r_safe = xr.where(M_r <= 20, np.nan, M_r) # To make sure that there's no neg log 
    #P_r = 43.43 * (5.6348 - np.log(M_r_safe - 20))
    
    # Then fallback cleanly
    #P_r = xr.where(M_r <= 20, P_o, P_r)
    ### end Mina's additions
    
    ## Apply rain condition if precip is less than 2.8 then use yesterday's DC
    P_r = xr.where(r_o <= 1.5, P_o, P_r)
    P_r = xr.where(P_r < 0, 0, P_r)

    ########################################################################
    ### (16) Log drying rate (K)
    K = (
        1.894 * (T + 1.1) * (100 - H) * (L_e * 1e-4)
    )  ## NOTE they use 1e-04 in the R but in the paper is 1e-06 code not sure what to use.

    ########################################################################
    ### (17) Duff moisture
    P = P_r + K

    # P = xr.DataArray(P, name="P", dims=("temp", "wind", "rh", "precip"))
    
    #ds["P"] = P
    #return ds
    return P


"""########################################################################"""
""" ############################ Drought Code #############################"""
"""########################################################################"""


def solve_dc(ds, D, L_f):

    ### Call on initial conditions
    #W, T, H, r_o, D_o, L_f = (ds.W, ds.T, ds.H, ds.r_o, D, L_f)
    #W, T, H, r_o, D_o, L_f = (ds.ws, ds.tC, ds.rh, ds.tp, D, L_f)
    W, T, H, r_o, D_o, L_f = (ds.sfcWind, ds.tasmax, ds.hurs, ds.pr, D, L_f.dryingfactor)


    #~
    # Make sure no RH > 100 or RH < 0
    H = xr.where(ds.hurs > 100, 100, xr.where(ds.hurs < 0, 0, ds.hurs))
    
    # Constrain temperature (Van Wagner 1985)
    #T = np.where(T < (-2.8), -2.8, T)
    T = xr.where(T < -2.8, -2.8, T)

    ########################################################################
    ### (18) Solve for the effective rain (r_d)
    r_d = 0.83 * r_o - 1.27

    ########################################################################
    ### (19) Solve for initial moisture equivalent (Q_o)
    Q_o = 800 * np.exp(-1 * D_o / 400)

    ########################################################################
    ### (20) Solve for moisture equivalent (Q_r)
    Q_r = Q_o + 3.937 * r_d

    ########################################################################
    ### (21) Solve for DC after rain (D_r)
    ## Alteration to Eq. 21 (Lawson 2008)
    D_r = D_o - 400 * np.log(1 + 3.937 * r_d / Q_o)
    # D_r = 400 * np.log(800 / Q_r)
    D_r = xr.where(D_r < 0, 0.0, D_r)
    D_r = xr.where(r_o <= 2.8, D_o, D_r)

    ########################################################################
    ### (22) Solve for potential evapotranspiration (V)
    # V = (0.36 * (T + 2.8)) + L_f
    V = (0.36 * (T + 2.8)) + L_f
    V = xr.where(V < 0, 0.0, V)

    ########################################################################
    ## Alteration to Eq. 23 (Lawson 2008)
    D = D_r + V * 0.5
    # D = xr.DataArray(D, name="D", dims=("temp", "wind", "rh", "precip"))
    D = xr.where(D < 15, 15, D)
    
    #ds["D"] = D
    #return ds
    return D


"""########################################################################"""
""" #################### Initial Spread Index #############################"""
"""########################################################################"""


def solve_isi(ds, ffmc, fbp=False):
    ### Call on initial conditions
    #W, F = ds.W, ds.F
    W = ds.sfcWind
    F = ffmc

    m_o = 147.2 * (101 - F) / (59.5 + F)

    ########################################################################
    ### (24) Solve for wind function (f_W) with condition for fbp
    f_W = xr.where(
        (W >= 40) & (fbp == True),
        12 * (1 - np.exp(-0.0818 * (W - 28))),
        np.exp(0.05039 * W),
    )

    ########################################################################
    ### (25) Solve for fine fuel moisture function (f_F)
    f_F = 91.9 * np.exp(-0.1386 * m_o) * (1 + np.power(m_o, 5.31) / 4.93e7)

    ########################################################################
    ### (26) Solve for initial spread index (R)
    R = 0.208 * f_W * f_F
    # R = xr.DataArray(R, name="R", dims=("temp", "wind", "rh", "precip"))
   
    #ds["R"] = R
    #return ds
    return R


"""########################################################################"""
""" ########################## Build up Index #############################"""
"""########################################################################"""


def solve_bui(dmc, dc):

    ### Call on initial conditions
    #P, D = ds.P, ds.D
    #zero_full = np.zeros_like(ds.T)

    P = dmc.dmc
    D = dc.dc
    zero_full = np.zeros_like(dmc.dmc)

    ########################################################################
    ### (27a and 27b) Solve for build up index where P =< 0.4D (U_a)
    U_low = xr.where(P <= 0.4 * D, 0.8 * P * D / (P + (0.4 * D)), zero_full)

    U_high = xr.where(
        P > 0.4 * D,
        P - (1 - 0.8 * D / (P + (0.4 * D))) * (0.92 + np.power((0.0114 * P), 1.7)),
        zero_full,
    )

    U = U_low + U_high
    U = xr.where(U < 0, 1.0, U)
    # U = xr.DataArray(U, name="U", dims=("temp", "wind", "rh", "precip"))
    
    #ds["U"] = U
    #return ds
    return U


"""########################################################################"""
""" ###################### Fire Weather Index #############################"""
"""########################################################################"""


def solve_fwi(isi, bui):

    ### Call on initial conditions

    #U, R = ds.U, ds.R
    U = bui.bui
    R = isi.isi

    ########################################################################
    ### (28) Solve for duff moisture function where U =< 80(f_D_a)
    f_D = xr.where(
        U > 80,
        1000 / (25 + 108.64 * np.exp(-0.023 * U)),
        (0.626 * np.power(U, 0.809)) + 2,
    )

    ########################################################################
    # (29) Solve for intermitate FWI
    B = 0.1 * R * f_D

    ########################################################################
    ### (30) Solve FWI
    S = xr.where(B <= 1, B, np.exp(2.72 * np.power((0.434 * np.log(B)), 0.647)))
    # S = xr.DataArray(S, name="S", dims=("temp", "wind", "rh", "precip"))
    #ds["S"] = S

    return S

    ########################################################################
    ### (31) Solve for daily severity rating (DSR)
    #DSR = 0.0272 * np.power(S, 1.77)
    ## DSR = xr.DataArray(DSR, name="DSR", dims=("temp", "wind", "rh", "precip"))
    
    #ds["DSR"] = DSR
    #return ds


In [134]:
### Calculate all FWI components
    # Ref for startup values: 
        # https://journals.ametsoc.org/view/journals/wefo/39/6/WAF-D-23-0226.1.xml 
    # Refs for applying functions to xarray:
        # https://stackoverflow.com/questions/57552588/apply-function-along-time-dimension-of-xarray 

# Default startup values for all components:
FFMCStart = 85
DMCStart = 6
DCStart = 15

# Note: Use a spinup of one year

In [135]:
# Try to initialize without the time dimension
ffmc00 = xr.DataArray(data = FFMCStart, dims=["lat","lon"], 
                     coords={"lat": climatexdata.lat, "lon": climatexdata.lon})
dmc00 = xr.DataArray(data = DMCStart, dims=["lat","lon"], 
                     coords={"lat": climatexdata.lat, "lon": climatexdata.lon})
dc00 = xr.DataArray(data = DCStart, dims=["lat","lon"], 
                     coords={"lat": climatexdata.lat, "lon": climatexdata.lon})

In [136]:
# Ensure all the xarray datasets have the same order of dimensions
climatexdata = climatexdata.transpose("time", "lat", "lon")
df_all = df_all.transpose("time", "lat", "lon")
daylength_all = daylength_all.transpose("time", "lat", "lon")

In [ ]:
# Now try: using Chris' FWI code, loop only through time
# Create xarray of previous time step with FFMC prev at all lat/lons
# Then loop through the function at each time step

ffmc_prev = ffmc00 ## set up first time step data array 

time = climatexdata['time'].values
ffmc_list = [] # To store the result at each time step

for i, time in enumerate(tqdm(time)):    
#for i, time in enumerate(tqdm(local_time[1:2])):    
    ds = climatexdata.isel(time=i)
    ffmcdata = solve_ffmc(ds, ffmc_prev) # Returns "F" - FFMC dataset for that timestep 
    ffmc_prev = ffmcdata # Set new FFMCPrevious for next time step
    
    # Drop problematic coord if present
    if "valid_time" in ffmcdata.coords:
        ffmcdata = ffmcdata.drop_vars("valid_time")
        
    ffmc_list.append(ffmcdata.expand_dims(time=[climatexdata.time.values[i]]))  # Match dim for concat
    
# Combine after the loop
ffmc_all = xr.concat(ffmc_list, dim="time").rename("ffmc").to_dataset()   # Run this line if using startup values
#ffmc_all = xr.concat(ffmc_list, dim="time")   # Run this line if continuing from known values 

# Clear encoding metadata to avoid Zarr chunk conflicts
for var in ffmc_all.variables:
    ffmc_all[var].encoding = {}

# Save to zarr
ffmc_all.to_zarr("CanESM5_ffmc_2004-2014.zarr", mode="w")


 98%|█████████▊| 3926/4015 [9:27:02<25:20, 17.09s/it]  

In [ ]:
ffmc_all = xr.open_zarr(folder, 'CanESM5_ffmc_2004-2014.zarr') 
ffmc_all

In [ ]:
ffmc_all.isel(time=3000).ffmc.values

In [ ]:
# Check plot
buiplot = ffmc_all.isel(time=3000)

# Plot sample ffmc
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy

fig = plt.figure(1, figsize=[30,13])

ax = plt.subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.coastlines()
ax.add_feature(cartopy.feature.BORDERS, linestyle='-', alpha=1)
ax.set_extent([-140, -113, 48, 60])

resol = '50m'

provinc_bodr = cartopy.feature.NaturalEarthFeature(category='cultural', 
    name='admin_1_states_provinces_lines', scale=resol, facecolor='none', edgecolor='k')
ax.add_feature(provinc_bodr, linestyle='--', linewidth=0.6, edgecolor="k", zorder=10)

m = buiplot.ffmc.plot(x="lon", y="lat", ax=ax, cmap='jet', add_colorbar=False)
ax.set_title("Daily FFMC Values - April 30, 2004", fontsize=35)

# Set up your own colorbar in order to adjust label size
    # Ref: https://github.com/pydata/xarray/issues/3275
cb = plt.colorbar(m)
cb.set_label(label='FFMC', fontsize=30)
cb.ax.tick_params(labelsize=30)


In [ ]:
# DMC Calculation

dmc_prev = dmc00 # set up first time step data array
dmc_list = [] # To store the result at each time step 
time = climatexdata['time'].values

for i, time in enumerate(tqdm(time)):    
#for i, time in enumerate(tqdm(time[1:3])):    
    ds = climatexdata.isel(time=i) # isolate this time step
    L_e = daylength_all.isel(time=i) # repeat for the daylength 

    dmcdata = solve_dmc(ds, dmc_prev, L_e) # Returns "P" - DMC dataset for that timestep 
    dmc_prev = dmcdata # Set new FFMCPrevious for next time step

     # Drop problematic coord if present
    if "valid_time" in dmcdata.coords:
        dmcdata = dmcdata.drop_vars("valid_time")
        
    dmc_list.append(dmcdata.expand_dims(time=[climatexdata.time.values[i]]))  # Match dim for concat
    
# Combine after the loop
#dmc_all = xr.concat(dmc_list, dim="time").rename({"daylength": "dmc"})
dmc_all = xr.concat(dmc_list, dim="time")

# To dataset
dmc_ds = dmc_all.to_dataset(name="dmc")

# Rechunk to match eradata dimensions
dmc_rechunk = dmc_ds.chunk({"time": 365, "lat": 5, "lon": 5})

# Clear encoding metadata to avoid Zarr chunk conflicts
for var in dmc_rechunk.variables:
    dmc_rechunk[var].encoding = {}

# Save to zarr
dmc_rechunk.to_zarr("CanESM5_dmc_2004-2014.zarr", mode="w")

In [ ]:
dmc_all = xr.open_zarr(folder, 'CanESM5_dmc_2004-2014.zarr') 
dmc_all

In [ ]:
dmc_all.isel(time=300).dmc.values

In [ ]:
# Check plot
buiplot = dmc_all.isel(time=300)

# Plot sample ffmc
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy

fig = plt.figure(1, figsize=[30,13])

ax = plt.subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.coastlines()
ax.add_feature(cartopy.feature.BORDERS, linestyle='-', alpha=1)
ax.set_extent([-140, -113, 48, 60])

resol = '50m'

provinc_bodr = cartopy.feature.NaturalEarthFeature(category='cultural', 
    name='admin_1_states_provinces_lines', scale=resol, facecolor='none', edgecolor='k')
ax.add_feature(provinc_bodr, linestyle='--', linewidth=0.6, edgecolor="k", zorder=10)

m = buiplot.dmc.plot(x="lon", y="lat", ax=ax, cmap='jet', add_colorbar=False)
ax.set_title("Daily DMC Values - April 30, 2004", fontsize=35)

# Set up your own colorbar in order to adjust label size
    # Ref: https://github.com/pydata/xarray/issues/3275
cb = plt.colorbar(m)
cb.set_label(label='DMC', fontsize=30)
cb.ax.tick_params(labelsize=30)

In [ ]:
# DC Calculation

dc_prev = dc00 # set up first time step data array
dc_list = [] # To store the result at each time step 
time = climatexdata['time'].values

for i, time in enumerate(tqdm(time)):  
#for i, time in enumerate(tqdm(time[1:5])):    
    ds = climatexdata.isel(time=i) # isolate this time step
    L_f = df_all.isel(time=i) # repeat for the daylength 

    dcdata = solve_dc(ds, dc_prev, L_f) # Returns "D" - DC dataset for that timestep 
    dc_prev = dcdata # Set new DCPrevious for next time step
    
    # Drop problematic coord if present
    if "valid_time" in dcdata.coords:
        dcdata = dcdata.drop_vars("valid_time")
        
    dc_list.append(dcdata.expand_dims(time=[climatexdata.time.values[i]]))  # Match dim for concat
    
# Combine after the loop
#dc_all = xr.concat(dc_list, dim="time").rename({"dryingfactor": "dc"})
dc_all = xr.concat(dc_list, dim="time")

# To dataset
dc_ds = dc_all.to_dataset(name="dc")

# Rechunk to match eradata dimensions
dc_rechunk = dc_ds.chunk({"time": 365, "lat": 5, "lon": 5})

# Clear encoding metadata to avoid Zarr chunk conflicts
for var in dc_rechunk.variables:
    dc_rechunk[var].encoding = {}

# Save to zarr
dc_rechunk.to_zarr("CanESM5_dc_2004-2014.zarr", mode="w")

In [ ]:
dc_all = xr.open_zarr(folder, 'CanESM5_dc_2004-2014.zarr') 
dc_all

In [ ]:
dc_all.isel(time=300).dc.values

In [ ]:
# Check plot
buiplot = dc_all.isel(time=1500)

# Plot sample ffmc
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy

fig = plt.figure(1, figsize=[30,13])

ax = plt.subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.coastlines()
ax.add_feature(cartopy.feature.BORDERS, linestyle='-', alpha=1)
ax.set_extent([-140, -113, 48, 60])

resol = '50m'

provinc_bodr = cartopy.feature.NaturalEarthFeature(category='cultural', 
    name='admin_1_states_provinces_lines', scale=resol, facecolor='none', edgecolor='k')
ax.add_feature(provinc_bodr, linestyle='--', linewidth=0.6, edgecolor="k", zorder=10)

m = buiplot.dc.plot(x="lon", y="lat", ax=ax, cmap='jet', add_colorbar=False)
ax.set_title("Daily FFMC Values - April 30, 2004", fontsize=35)

# Set up your own colorbar in order to adjust label size
    # Ref: https://github.com/pydata/xarray/issues/3275
cb = plt.colorbar(m)
cb.set_label(label='FFMC', fontsize=30)
cb.ax.tick_params(labelsize=30)

In [ ]:
# Load FFMC, DMC, DC for next calculations
ffmc_all = xr.open_zarr(r"C:\Users\minal\Documents\UBC\GRID2\CanESM5_ffmc_2004-2014.zarr")
dmc_all = xr.open_zarr(r"C:\Users\minal\Documents\UBC\GRID2\CanESM5_dmc_2004-2014.zarr")
dc_all = xr.open_zarr(r"C:\Users\minal\Documents\UBC\GRID2\CanESM5_dc_2004-2014.zarr")

In [ ]:
# ISI Calculation

isi_list = [] # To store the result at each time step 
time = climatexdata['time'].values

for i, time in enumerate(tqdm(time)):    
#for i, time in enumerate(tqdm(local_time[10:12])):    
    ds = climatexdata.isel(time=i) # isolate this time step
    ffmc = ffmc_all.isel(time=i) # repeat for the daylength 

    isidata = solve_isi(ds, ffmc) # Returns "R" - ISI dataset for that timestep 
    isi_list.append(isidata.expand_dims(time=[time]))   # Match dim for concat

# Combine after the loop
isi_all = xr.concat(isi_list, dim="time") #.to_dataset(name="dc")

# Explicitly set the variable name to 'isi'
isi_all = isi_all.rename({'ffmc': 'isi'})  # Rename the concatenated variable to 'isi'

# Rechunk to match eradata dimensions
isi_rechunk = isi_all.chunk({"time": 365, "lat": 5, "lon": 5})

# Clear encoding metadata to avoid Zarr chunk conflicts
for var in isi_rechunk.variables:
    isi_rechunk[var].encoding = {}

# Save to zarr
isi_rechunk.to_zarr("CanESM5_isi_2004-2014.zarr", mode="w")

In [ ]:
# BUI Calculation

bui_list = [] # To store the result at each time step 
time = climatexdata['time'].values

for i, time in enumerate(tqdm(time)):    
#for i, time in enumerate(tqdm(local_time[10:12])):    
    dmc_i = dmc_all.isel(time=i) # isolate this time step
    dc_i = dc_all.isel(time=i) # repeat for the daylength 

    buidata = solve_bui(dmc_i, dc_i) # Returns "U" - BUI dataset for that timestep 
    buidata.name = 'bui' #Need to name the variable
    bui_list.append(buidata.expand_dims(time=[time]))   # Match dim for concat

# Combine after the loop
bui_all = xr.concat(bui_list, dim="time") 

# Convert from dataarray to dataset
bui_all = xr.Dataset({"bui": bui_all})

# Rechunk to match eradata dimensions
bui_rechunk = bui_all.chunk({"time": 365, "lat": 5, "lon": 5})

# Clear encoding metadata to avoid Zarr chunk conflicts
for var in bui_rechunk.variables:
    bui_rechunk[var].encoding = {}

# Save to zarr
bui_rechunk.to_zarr("CanESM5_bui_2004-2014.zarr", mode="w")

In [ ]:
# Load BUI and ISI for FWI calculation
isi_all = xr.open_zarr(r"C:\Users\minal\Documents\UBC\GRID2\CanESM5_isi_2004-2014.zarr")
bui_all = xr.open_zarr(r"C:\Users\minal\Documents\UBC\GRID2\CanESM5_bui_2004-2014.zarr")

In [ ]:
isi_all.isel(time=300).isi.values

In [ ]:
bui_all.isel(time=300).bui.values

In [ ]:
# FWI Calculation

fwi_list = [] # To store the result at each time step 
time = climatexdata['time'].values

for i, time in enumerate(tqdm(time)):    
#for i, time in enumerate(tqdm(local_time[10:12])):    
    isi_i = isi_all.isel(time=i) # isolate this time step
    bui_i = bui_all.isel(time=i) # repeat for the daylength 

    fwidata = solve_fwi(isi_i, bui_i) 
    fwidata.name = 'fwi' #Need to name the variable
    fwi_list.append(fwidata.expand_dims(time=[time]))   # Match dim for concat

# Combine after the loop
fwi_all = xr.concat(fwi_list, dim="time") 

# Convert from dataarray to dataset
fwi_all = xr.Dataset({"fwi": fwi_all})

# Rechunk to match eradata dimensions
fwi_rechunk = fwi_all.chunk({"time": 365, "lat": 5, "lon": 5})

# Clear encoding metadata to avoid Zarr chunk conflicts
for var in fwi_rechunk.variables:
    fwi_rechunk[var].encoding = {}

# Save to zarr
fwi_rechunk.to_zarr("CanESM5_fwi_2004-2014.zarr", mode="w")

In [ ]:
fwi_all = xr.open_zarr(r"C:\Users\minal\Documents\UBC\GRID2\CanESM5_fwi_2004-2014.zarr")

In [ ]:
# Check plot
buiplot = fwi_all.isel(time=3470)

# Plot sample ffmc
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy

fig = plt.figure(1, figsize=[30,13])

ax = plt.subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.coastlines()
ax.add_feature(cartopy.feature.BORDERS, linestyle='-', alpha=1)
ax.set_extent([-140, -113, 48, 60])

resol = '50m'

provinc_bodr = cartopy.feature.NaturalEarthFeature(category='cultural', 
    name='admin_1_states_provinces_lines', scale=resol, facecolor='none', edgecolor='k')
ax.add_feature(provinc_bodr, linestyle='--', linewidth=0.6, edgecolor="k", zorder=10)

m = buiplot.fwi.plot(x="lon", y="lat", ax=ax, cmap='jet', add_colorbar=False)
ax.set_title("Daily FWI Values - April 30, 2004", fontsize=35)

# Set up your own colorbar in order to adjust label size
    # Ref: https://github.com/pydata/xarray/issues/3275
cb = plt.colorbar(m)
cb.set_label(label='FWI', fontsize=30)
cb.ax.tick_params(labelsize=30)